# SAM_demo_AIS Notebook

## Artificial Immune System (AIS) Optimization for Microbolometer Sensor Design

This notebook implements an AIS algorithm to optimize basis functions for maximum substance separability, using the same framework as the GA implementation but with immune system-inspired operations.


<h2 id="data-loading-and-preprocessing">1. Data Loading and Preprocessing</h2>


In [ ]:
# Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm

import os
import sys

sys.path.insert(0, os.path.abspath(".."))
from tools import (
    simulate_sensor_output,
    compute_distance_matrix,
    spectral_angle_mapper,
    min_based_dissimilarity_score,
)

In [ ]:
# Load substances and generate basis functions

# --------------------------------------------------------------------------------------------------
# File Locations
# --------------------------------------------------------------------------------------------------

# File paths for input data
spectral_data_file = "../data/Test 3 - 4 White Powers/white_powders_with_labels.xlsx"
air_transmittance_file = "../data/Test 3 - 4 White Powers/Air transmittance.xlsx"
basis_functions_file = "../data/Test 3 - 4 White Powers/Basis functions_4-20um.xlsx"

# --------------------------------------------------------------------------------------------------
# Configurable Parameters
# --------------------------------------------------------------------------------------------------

atmospheric_distance_ratio = (
    0.11  # Used to model the effect of atmospheric conditions on measurements
)
temperature_K = 293.15  # Temperature in Kelvin
air_refractive_index = 1  # Refractive index of air

# --------------------------------------------------------------------------------------------------
# Loading Spectral Data & Extracting Features
# --------------------------------------------------------------------------------------------------

# Load the spectral data of substances
substances_spectral_data = pd.read_excel(spectral_data_file)

# Verify the dimensions of the loaded data
# print("Substances spectral data shape:", substances_spectral_data.shape)

# Extract the wavelength values (first column)
wavelengths = substances_spectral_data.iloc[:, :1].to_numpy()  # Shape = (d, 1)
# print(wavelengths.iloc[:, 0].tolist())

# Extract substance labels (column names excluding the first)
substance_names = substances_spectral_data.columns[1:].to_numpy()

# Extract emissivity curves (all data except the first column)
emissivity_curves = substances_spectral_data.iloc[:, 1:].to_numpy()
# print("Emissivity curves shape:", emissivity_curves.shape)

# --------------------------------------------------------------------------------------------------
# Loading Additional Parameters
# --------------------------------------------------------------------------------------------------

# Load air transmittance matrix
air_transmittance = np.array(pd.read_excel(air_transmittance_file, header=None))
air_transmittance = air_transmittance[:, 1:]  # Remove the spectra column

# --------------------------------------------------------------------------------------------------
# Visualization: Spectral Emissivity Curves
# --------------------------------------------------------------------------------------------------

# Plot the emissivity curves for all substances
plt.figure(figsize=(6, 4))
for name in substance_names:
    plt.plot(wavelengths, substances_spectral_data[name], label=name)

# Add plot labels and grid
plt.xlabel("Wavelength (µm)")
plt.ylabel("Emissivity Value")
# plt.title('Spectral Emissivity Curves of Substances')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

<h2 id="ais-implementation">2. AIS Algorithm Implementation</h2>


In [ ]:
# Chromosome to basis functions with peak alignment
# ==================================================================
import numpy as np


def generate_basis_from_chromosome(chromosome, wavelengths):
    """
    Translates a GA chromosome into a set of Gaussian basis function curves,
    aligning each mean to the closest discrete wavelength to ensure a peak of 1.0.

    Parameters:
    - chromosome (list or np.ndarray): An 8-element list of [µ1, σ1, µ2, σ2, ...].
    - wavelengths (np.ndarray): A 1D array of discrete wavelength values for the x-axis.

    Returns:
    - np.ndarray: An array of shape (num_wavelengths, 4) where each column
                  is a generated and aligned Gaussian basis function.
    """
    # Ensure wavelengths is a column vector for broadcasting
    if wavelengths.ndim == 1:
        wavelengths = wavelengths.reshape(-1, 1)

    num_basis_funcs = 4
    num_wavelengths = len(wavelengths)
    basis_functions = np.zeros((num_wavelengths, num_basis_funcs))

    for i in range(num_basis_funcs):
        # Extract the raw mean (µ) and sigma (σ) for the current basis function
        raw_mean = chromosome[i * 2]
        sigma = chromosome[i * 2 + 1]

        # ==================================================================
        # Added Logic: Align the raw mean to the closest actual wavelength
        # ==================================================================
        aligned_mean = wavelengths[np.abs(wavelengths - raw_mean).argmin()][0]

        # Generate the Gaussian curve using the aligned mean
        # f(x) = exp(- (x - µ_aligned)² / (2 * σ²) )
        exponent = -((wavelengths - aligned_mean) ** 2) / (2 * sigma**2)
        basis_functions[:, i] = np.exp(exponent).flatten()

    return basis_functions

In [ ]:
# Fitness Function (same as GA)
# ==================================================================


def fitness_func(chromosome):
    """
    Calculates the fitness of a given chromosome of µ and σ parameters.
    The fitness is determined by the min-based dissimilarity score.
    """
    # Step 1: Generate the basis function curves directly from the chromosome.
    current_basis_funcs = generate_basis_from_chromosome(chromosome, wavelengths)

    # Step 2: Simulate the sensor output using the generated basis functions
    sensor_outputs = simulate_sensor_output(
        wavelengths=wavelengths,
        substances_emissivity=emissivity_curves,
        basis_functions=current_basis_funcs,
        temperature_K=temperature_K,
        atmospheric_distance_ratio=atmospheric_distance_ratio,
        air_refractive_index=air_refractive_index,
        air_transmittance=air_transmittance,
    )

    # Step 3: Compute the distance matrix from the sensor outputs
    distance_matrix = compute_distance_matrix(sensor_outputs, spectral_angle_mapper)

    # Step 4: Calculate the dissimilarity score, which will be our fitness value.
    # We want to MAXIMIZE this score.
    fitness = min_based_dissimilarity_score(distance_matrix)

    return fitness

In [ ]:
# Distance function for antibody similarity (same as GA)
# ==================================================================
# This function ensures consistent distance calculation between GA and AIS
# It handles the fact that parameter pair order doesn't matter (e.g., [1,2,3,4] ≡ [1,3,2,4])


def distance_optimal_pairing(chromosome_a, chromosome_b):
    """
    Computes the distance between two chromosomes using optimal pairing of their
    basis functions (genes). This is robust to the ordering of genes.
    """
    # Step 1: Reshape 1D gene arrays into 2D sets of (mean, sigma) pairs
    genes_a = chromosome_a.reshape(-1, 2)
    genes_b = chromosome_b.reshape(-1, 2)

    # Step 2: Compute the pairwise distance matrix (cost matrix)
    num_basis_functions = genes_a.shape[0]
    pairwise_distances = np.zeros((num_basis_functions, num_basis_functions))

    for i in range(num_basis_functions):
        for j in range(num_basis_functions):
            # Euclidean distance between two (mean, sigma) pairs
            pairwise_distances[i, j] = np.linalg.norm(genes_a[i] - genes_b[j])

    # Step 3: Find optimal pairing using the Hungarian algorithm
    row_indices, col_indices = linear_sum_assignment(pairwise_distances)

    # Step 4: Sum the distances of the optimal pairs
    return pairwise_distances[row_indices, col_indices].sum()

In [ ]:
# AIS Algorithm Implementation
# ==================================================================


class Antibody:
    """
    Represents an antibody (solution) in the AIS algorithm.
    """

    def __init__(self, genes, fitness=None):
        self.genes = np.array(genes)
        self.fitness = fitness
        self.age = 0  # Age of the antibody

    def clone(self):
        """Create a copy of this antibody."""
        return Antibody(self.genes.copy(), self.fitness)


class AISOptimizer:
    """
    Artificial Immune System optimizer for basis function optimization.
    """

    def __init__(
        self,
        population_size=200,
        max_generations=1000,
        clone_factor=0.1,
        mutation_rate=0.1,
        suppression_threshold=0.1,
        memory_size=100,
        gene_bounds=None,
        random_seed=42,
    ):
        # AIS Parameters
        self.population_size = population_size
        self.max_generations = max_generations
        self.clone_factor = clone_factor
        self.mutation_rate = mutation_rate
        self.suppression_threshold = suppression_threshold
        self.memory_size = memory_size
        self.gene_bounds = gene_bounds

        # Initialize random seed
        np.random.seed(random_seed)

        # Population and memory
        self.population = []
        self.memory_cells = []

        # Statistics tracking
        self.best_fitness_per_gen = []
        self.worst_fitness_per_gen = []
        self.mean_fitness_per_gen = []
        self.fitness_snapshots = {}
        self.diversity_per_gen = []  # Track population diversity over generations

        # Generation counter
        self.generation = 0

    def initialize_population(self):
        """Initialize the population with random antibodies."""
        self.population = []

        for _ in range(self.population_size):
            # Generate random genes within bounds
            genes = []
            for bound in self.gene_bounds:
                genes.append(np.random.uniform(bound[0], bound[1]))

            antibody = Antibody(genes)
            antibody.fitness = fitness_func(antibody.genes)
            self.population.append(antibody)

    def clonal_selection(self):
        """
        Select antibodies for cloning based on their fitness (affinity).
        Higher fitness antibodies get more clones.
        """
        # Sort population by fitness (descending)
        sorted_pop = sorted(self.population, key=lambda x: x.fitness, reverse=True)

        # Calculate number of clones for each antibody
        clones = []

        for i, antibody in enumerate(sorted_pop):
            # Number of clones proportional to fitness rank
            num_clones = max(
                1, int(self.clone_factor * self.population_size * (1 - i / self.population_size))
            )

            for _ in range(num_clones):
                clone = antibody.clone()
                clones.append(clone)

        return clones

    def affinity_maturation(self, clones):
        """
        Mutate clones with mutation rate inversely proportional to fitness.
        """
        mutated_clones = []

        # Get fitness range for normalization
        fitnesses = [clone.fitness for clone in clones]
        min_fitness = min(fitnesses)
        max_fitness = max(fitnesses)
        fitness_range = max_fitness - min_fitness if max_fitness > min_fitness else 1

        for clone in clones:
            # Calculate adaptive mutation rate (inverse to fitness)
            normalized_fitness = (clone.fitness - min_fitness) / fitness_range
            adaptive_mutation_rate = self.mutation_rate * (1 - normalized_fitness)

            # Mutate genes
            mutated_genes = clone.genes.copy()

            for i, (gene, bound) in enumerate(zip(mutated_genes, self.gene_bounds)):
                if np.random.random() < adaptive_mutation_rate:
                    # Gaussian mutation
                    mutation_strength = (bound[1] - bound[0]) * 0.1  # 10% of range
                    noise = np.random.normal(0, mutation_strength)
                    mutated_genes[i] = np.clip(gene + noise, bound[0], bound[1])

            # Create new antibody with mutated genes
            mutated_antibody = Antibody(mutated_genes)
            mutated_antibody.fitness = fitness_func(mutated_antibody.genes)
            mutated_clones.append(mutated_antibody)

        return mutated_clones

    def suppression(self, antibodies):
        """
        Remove similar antibodies to maintain diversity.
        """
        if len(antibodies) <= self.population_size:
            return antibodies

        # Sort by fitness (descending)
        sorted_antibodies = sorted(antibodies, key=lambda x: x.fitness, reverse=True)

        # Keep the best antibody
        selected = [sorted_antibodies[0]]

        # Add antibodies that are sufficiently different from already selected ones
        for antibody in sorted_antibodies[1:]:
            is_different = True

            for selected_antibody in selected:
                distance = distance_optimal_pairing(antibody.genes, selected_antibody.genes)
                if distance < self.suppression_threshold:
                    is_different = False
                    break

            if is_different and len(selected) < self.population_size:
                selected.append(antibody)

        return selected

    def update_memory(self):
        """
        Update memory cells with best antibodies from current population.
        """
        # Get best antibodies from current population
        sorted_pop = sorted(self.population, key=lambda x: x.fitness, reverse=True)

        # Add to memory if better than worst memory cell or if memory not full
        for antibody in sorted_pop[:10]:  # Consider top 10
            if len(self.memory_cells) < self.memory_size:
                self.memory_cells.append(antibody.clone())
            else:
                # Replace worst memory cell if current is better
                worst_memory = min(self.memory_cells, key=lambda x: x.fitness)
                if antibody.fitness > worst_memory.fitness:
                    self.memory_cells.remove(worst_memory)
                    self.memory_cells.append(antibody.clone())

        # Sort memory by fitness
        self.memory_cells.sort(key=lambda x: x.fitness, reverse=True)

    def run_generation(self):
        """
        Run one generation of the AIS algorithm.
        """
        # Step 1: Clonal Selection
        clones = self.clonal_selection()

        # Step 2: Affinity Maturation (mutation)
        mutated_clones = self.affinity_maturation(clones)

        # Step 3: Combine original population with mutated clones
        combined_population = self.population + mutated_clones

        # Step 4: Suppression (diversity maintenance)
        self.population = self.suppression(combined_population)

        # Step 5: Update memory cells
        self.update_memory()

        # Step 6: Update statistics
        self.update_statistics()

        self.generation += 1

    def calculate_diversity(self):
        """
        Calculate population diversity as the mean pairwise distance between all antibodies.
        """
        if len(self.population) < 2:
            return 0.0

        total_distance = 0.0
        count = 0

        for i in range(len(self.population)):
            for j in range(i + 1, len(self.population)):
                distance = distance_optimal_pairing(
                    self.population[i].genes, self.population[j].genes
                )
                total_distance += distance
                count += 1

        return total_distance / count if count > 0 else 0.0

    def update_statistics(self):
        """
        Update generation statistics.
        """
        fitnesses = [antibody.fitness for antibody in self.population]

        self.best_fitness_per_gen.append(max(fitnesses))
        self.worst_fitness_per_gen.append(min(fitnesses))
        self.mean_fitness_per_gen.append(np.mean(fitnesses))
        self.diversity_per_gen.append(self.calculate_diversity())

        # Take snapshots at intervals
        if self.generation % 500 == 0 and self.generation > 0:
            self.fitness_snapshots[self.generation] = fitnesses.copy()

    def run(self, progress_callback=None):
        """
        Run the complete AIS optimization.
        """
        print("Initializing AIS population...")
        self.initialize_population()

        print(f"Running AIS optimization for {self.max_generations} generations...")

        for gen in tqdm(range(self.max_generations), desc="AIS Evolution"):
            self.run_generation()

            if progress_callback:
                progress_callback(self)

        print("AIS optimization complete!")

    def get_best_solution(self):
        """
        Get the best solution found.
        """
        best_antibody = max(self.population, key=lambda x: x.fitness)
        return best_antibody.genes, best_antibody.fitness

    def get_memory_cells(self):
        """
        Get all memory cells.
        """
        return self.memory_cells

<h2 id="ais-configuration-and-run">3. AIS Configuration and Run</h2>


In [ ]:
# AIS Configuration
# ==================================================================
# --- Define parameter bounds (same as GA) ---
MU_MIN, MU_MAX = 4.0, 20.0
SIGMA_MIN, SIGMA_MAX = 0.1, 4.0

# --- Define the gene bounds for AIS ---
# Gene bounds is a list of (min, max) tuples for each of the 8 genes.
# Genes 0, 2, 4, 6 are means (µ)
# Genes 1, 3, 5, 7 are standard deviations (σ)
gene_bounds = []
for i in range(4):
    gene_bounds.append((MU_MIN, MU_MAX))  # Space for µ
    gene_bounds.append((SIGMA_MIN, SIGMA_MAX))  # Space for σ

# --- AIS Parameters ---
POPULATION_SIZE = 200
NUM_GENERATIONS = 200
CLONE_FACTOR = 0.1  # Fraction of population to clone
MUTATION_RATE = 0.1  # Base mutation rate
SUPPRESSION_THRESHOLD = (
    0.8  # Distance threshold for suppression (balanced for diversity + convergence)
)
MEMORY_SIZE = 100  # Size of memory cell population
RANDOM_SEED = 42

# --- Progress tracking setup ---
generational_bests = []
fitness_snapshots = {}

# --- Create a progress bar instance ---
pbar = tqdm(total=NUM_GENERATIONS, desc="AIS Evolution")


def on_generation(ais_instance):
    """
    Callback function to perform tasks after each generation:
    1. Track generational champions.
    2. Update the progress bar.
    """
    global generational_bests, fitness_snapshots

    # --- Generational Champions Logic ---
    gen_num = ais_instance.generation
    if (gen_num % 20 == 0) and (0 < gen_num <= 200):
        best_genes, best_fitness = ais_instance.get_best_solution()
        generational_bests.append((gen_num, best_fitness, best_genes.copy()))

    # --- Take a snapshot of the full population's fitness every 500 generations ---
    if (gen_num % 500 == 0) and (gen_num > 0):
        fitnesses = [antibody.fitness for antibody in ais_instance.population]
        fitness_snapshots[gen_num] = fitnesses.copy()

    # --- Progress Bar Update Logic ---
    pbar.update(1)


print("AIS Configuration Complete!")
print(f"Population Size: {POPULATION_SIZE}")
print(f"Generations: {NUM_GENERATIONS}")
print(f"Clone Factor: {CLONE_FACTOR}")
print(f"Mutation Rate: {MUTATION_RATE}")
print(f"Suppression Threshold: {SUPPRESSION_THRESHOLD}")
print(f"Memory Size: {MEMORY_SIZE}")

In [ ]:
# Run AIS Optimization
# ==================================================================
print("Running AIS Optimization... 🧬")

# Create AIS optimizer instance
ais_optimizer = AISOptimizer(
    population_size=POPULATION_SIZE,
    max_generations=NUM_GENERATIONS,
    clone_factor=CLONE_FACTOR,
    mutation_rate=MUTATION_RATE,
    suppression_threshold=SUPPRESSION_THRESHOLD,
    memory_size=MEMORY_SIZE,
    gene_bounds=gene_bounds,
    random_seed=RANDOM_SEED,
)

# Run the optimization
ais_optimizer.run(progress_callback=on_generation)

# Close the progress bar
pbar.close()
print("AIS optimization complete!")

<h2 id="ais-results-analysis">4. AIS Results Analysis and Visualization</h2>


In [ ]:
# Get Best Solution and Print Results
# ==================================================================

# --- Retrieve the best solution ---
best_genes, best_fitness = ais_optimizer.get_best_solution()
print("\n🎯 AIS Best Solution Found:")
print("Best solution (µ, σ pairs):")
for i in range(4):
    print(f"  Basis Function {i + 1}: µ = {best_genes[i * 2]:.3f}, σ = {best_genes[i * 2 + 1]:.3f}")
print(f"Fitness value of the best solution = {best_fitness:.4f}")

# --- Get final population fitness values ---
final_fitness = [antibody.fitness for antibody in ais_optimizer.population]
sorted_fitness = np.sort(final_fitness)[::-1]

print("\n📊 AIS Final Population Statistics:")
print(f"Best fitness: {max(final_fitness):.4f}")
print(f"Worst fitness: {min(final_fitness):.4f}")
print(f"Mean fitness: {np.mean(final_fitness):.4f}")
print(f"Std fitness: {np.std(final_fitness):.4f}")

# --- Memory cells statistics ---
memory_cells = ais_optimizer.get_memory_cells()
if memory_cells:
    memory_fitnesses = [cell.fitness for cell in memory_cells]
    print("\n🧠 Memory Cells Statistics:")
    print(f"Number of memory cells: {len(memory_cells)}")
    print(f"Best memory fitness: {max(memory_fitnesses):.4f}")
    print(f"Mean memory fitness: {np.mean(memory_fitnesses):.4f}")

In [ ]:
# Plot Fitness Distribution of Final Generation
# ==================================================================

# --- Create the plot ---
plt.figure(figsize=(12, 7))
plt.bar(range(len(sorted_fitness)), sorted_fitness, color="lightcoral")
plt.title("AIS Fitness Distribution of the Final Generation", fontsize=16)
plt.xlabel("Individual (Sorted by Fitness)", fontsize=12)
plt.ylabel("Fitness Score", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Plot AIS Fitness Statistics Over Generations
# ==================================================================

# --- Plot Custom Fitness Statistics ---
plt.figure(figsize=(10, 6))
plt.plot(ais_optimizer.best_fitness_per_gen, label="Best Fitness", color="green")
plt.plot(ais_optimizer.mean_fitness_per_gen, label="Mean Fitness", color="blue")
plt.plot(ais_optimizer.worst_fitness_per_gen, label="Worst Fitness", color="red", linestyle="--")
plt.title("AIS Fitness Statistics Over Generations", fontsize=16)
plt.xlabel("Generation")
plt.ylabel("Fitness")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

# --- Plot Diversity Over Generations ---
plt.figure(figsize=(10, 6))
plt.plot(ais_optimizer.diversity_per_gen, label="Population Diversity", color="purple", linewidth=2)
plt.axhline(
    y=SUPPRESSION_THRESHOLD,
    color="orange",
    linestyle="--",
    label=f"Suppression Threshold ({SUPPRESSION_THRESHOLD})",
)
plt.title("AIS Population Diversity Over Generations", fontsize=16)
plt.xlabel("Generation")
plt.ylabel("Mean Pairwise Distance")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

print("\n📊 Diversity Analysis:")
print(f"Final population diversity: {ais_optimizer.diversity_per_gen[-1]:.3f}")
print(f"Suppression threshold: {SUPPRESSION_THRESHOLD}")
print(
    f"Diversity vs threshold ratio: {ais_optimizer.diversity_per_gen[-1] / SUPPRESSION_THRESHOLD:.2f}"
)

In [ ]:
# Plot Best Basis Functions Found by AIS
# ==================================================================

# --- Plot Best Basis Functions "Canvas" ---
best_basis_functions = generate_basis_from_chromosome(best_genes, wavelengths)

plt.figure(figsize=(10, 5))
for i in range(best_basis_functions.shape[1]):
    plt.plot(
        wavelengths,
        best_basis_functions[:, i],
        label=f"µ={best_genes[i * 2]:.2f}, σ={best_genes[i * 2 + 1]:.2f}",
    )

# Plot original substances in the background for context
for i, name in enumerate(substance_names):
    plt.plot(wavelengths, emissivity_curves[:, i], alpha=0.15)

plt.title("Best Basis Functions Found by AIS", fontsize=16)
plt.xlabel("Wavelength (µm)")
plt.ylabel("Intensity / Emissivity")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Plot Top 20 AIS Solutions (similar to GA's top 10)
# ==================================================================


def plot_top_n_ais_solutions(
    population, n=50, wavelengths=None, vertical_offset=0.2, figure_size=(10, 12)
):
    """
    Plots the top N AIS solutions, similar to the GA's top 10 plot.
    """
    # Sort by fitness (descending) and take the top N
    top_solutions = sorted(population, key=lambda x: x.fitness, reverse=True)[:n]

    num_solutions = len(top_solutions)
    if num_solutions == 0:
        print("No solutions to plot.")
        return

    colors = plt.cm.viridis_r(np.linspace(0, 1, num_solutions))
    plt.figure(figsize=figure_size)

    for i in reversed(range(num_solutions)):
        antibody = top_solutions[i]
        basis_funcs = generate_basis_from_chromosome(antibody.genes, wavelengths)
        rank = i

        for j, basis_func in enumerate(basis_funcs.T):
            scaled_basis_func = basis_func * 0.5  # Scale for better visualization
            plt.plot(
                wavelengths,
                scaled_basis_func + (num_solutions - 1 - rank) * vertical_offset,
                color=colors[rank],
                alpha=0.8,
                linewidth=2,
                label=f"Rank {rank + 1}" if j == 0 else None,
            )

    plt.title(f"Top {num_solutions} AIS Sensor Designs")
    plt.xlabel("Wavelength (µm)")
    plt.ylabel("Absorptivity (Offset Applied for Visualization)")

    # Create rank labels for y-axis
    rank_labels = [f"Rank {i + 1}" for i in range(num_solutions)]
    rank_offsets = [(num_solutions - 1 - i) * vertical_offset for i in range(num_solutions)]
    plt.yticks(rank_offsets, rank_labels)

    plt.grid(True, linestyle="--", alpha=0.5)

    # Add legend (reverse order to match plot order)
    handles, labels = plt.gca().get_legend_handles_labels()
    plt.legend(handles[::-1], labels[::-1], title="Sensor Ranking", loc="upper left")

    plt.tight_layout()
    plt.show()


# Check if ais_optimizer exists and plot top 20 AIS solutions
try:
    if "ais_optimizer" in globals() and ais_optimizer is not None:
        print("✅ AIS optimizer found! Plotting top 20 solutions...")
        plot_top_n_ais_solutions(
            ais_optimizer.population, n=50, wavelengths=wavelengths, vertical_offset=0.3
        )
    else:
        print("❌ AIS optimizer not found. Please run the AIS optimization cells first.")
except NameError:
    print("❌ AIS optimizer not found. Please run the AIS optimization cells first.")
    print("💡 Tip: Run cells 1-11 to initialize and run the AIS optimization.")

In [ ]:
# Plot Fitness Distribution Snapshots
# ==================================================================
plt.figure(figsize=(10, 7))
colors = plt.cm.viridis(np.linspace(0, 1, len(fitness_snapshots)))

# Iterate through the snapshots and plot each one
for i, (gen_num, fitness_scores) in enumerate(fitness_snapshots.items()):
    # Sort the scores for a clean line plot
    sorted_scores = sorted(fitness_scores, reverse=True)
    plt.plot(sorted_scores, color=colors[i], label=f"Generation {gen_num}")

plt.title("AIS Fitness Distribution at Different Generational Snapshots", fontsize=16)
plt.xlabel("Solution Index (Sorted by Fitness)")
plt.ylabel("Fitness Score")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Diversity Analysis Functions (adapted from GA)
# ==================================================================


def generate_distance_matrix(antibodies_list):
    """
    Generates a pairwise distance matrix for a list of antibodies.

    Args:
        antibodies_list (list): A list of Antibody objects.

    Returns:
        numpy.ndarray: A symmetric matrix of pairwise distances.
    """
    num_antibodies = len(antibodies_list)
    distance_matrix = np.zeros((num_antibodies, num_antibodies))

    for i in range(num_antibodies):
        for j in range(i, num_antibodies):  # Only compute the upper triangle
            if i == j:
                distance_matrix[i, j] = 0
            else:
                dist = distance_optimal_pairing(antibodies_list[i].genes, antibodies_list[j].genes)
                distance_matrix[i, j] = dist
                distance_matrix[j, i] = dist  # Matrix is symmetric

    return distance_matrix


def analyze_antibody_diversity(final_population, top_n, F_threshold, R_peak):
    """
    Analyzes the top N antibodies in a population to report on the diversity
    and quality of high-performing antibody families (clusters).

    Args:
        final_population (list): The list of Antibody objects in the final generation.
        top_n (int): The number of top antibodies to consider from the population.
        F_threshold (float): The minimum fitness value for an antibody to be included.
        R_peak (float): The distance radius for clustering.

    Returns:
        dict: A dictionary containing the analysis report.
    """
    # Step 1: Select and filter top, high-quality antibodies
    final_population.sort(key=lambda x: x.fitness, reverse=True)
    top_antibodies = final_population[:top_n]
    antibodies_high = [ab for ab in top_antibodies if ab.fitness >= F_threshold]

    if not antibodies_high:
        return {
            "total_antibodies_analyzed": len(top_antibodies),
            "high_quality_count": 0,
            "family_count": 0,
            "families": [],
        }

    # Step 2: Cluster antibodies into families
    antibody_families = []
    for ab in antibodies_high:
        found_family = False
        for family in antibody_families:
            if distance_optimal_pairing(ab.genes, family["representative"].genes) <= R_peak:
                family["members"].append(ab)
                if ab.fitness > family["top_score"]:
                    family["top_score"] = ab.fitness
                found_family = True
                break

        if not found_family:
            antibody_families.append(
                {"representative": ab, "members": [ab], "top_score": ab.fitness}
            )

    # Step 3: Prepare the final report
    antibody_families.sort(key=lambda x: x["top_score"], reverse=True)

    family_reports = []
    for i, family in enumerate(antibody_families):
        family_reports.append(
            {
                "family_id": i + 1,
                "top_score": family["top_score"],
                "member_count": len(family["members"]),
                "representative_antibody": family["representative"].genes,
            }
        )

    report = {
        "total_antibodies_analyzed": len(top_antibodies),
        "high_quality_count": len(antibodies_high),
        "family_count": len(antibody_families),
        "families": family_reports,
        "high_quality_antibodies": antibodies_high,
    }

    return report

In [ ]:
# VAT and iVAT Functions (same as GA)
# ==================================================================


def vat_reorder(distance_matrix):
    """
    Reorders a given distance matrix using the VAT method.
    """
    n = distance_matrix.shape[0]

    # Step 1: Find the most dissimilar pair (farthest apart)
    i, j = np.unravel_index(np.argmax(distance_matrix, axis=None), distance_matrix.shape)

    # Step 2: Start with one of the most distant points
    reorder = [i]
    remaining = set(range(n)) - {i}  # Remaining points to process

    # Step 3: Iteratively find the next closest point to any selected point
    while remaining:
        next_index = min(remaining, key=lambda x: min(distance_matrix[x, p] for p in reorder))
        reorder.append(next_index)
        remaining.remove(next_index)

    # Step 4: Apply the reordering to the distance matrix
    vat_matrix = distance_matrix[np.ix_(reorder, reorder)]

    return vat_matrix, reorder


def iVAT_transform(vat_matrix):
    """
    Compute the iVAT distance transform (path-based distances) from a VAT-reordered matrix.
    """
    N = vat_matrix.shape[0]
    ivat_matrix = np.zeros_like(vat_matrix)

    for r in range(1, N):
        # Find the closest connection j for row r among previously processed rows
        j = np.argmin(vat_matrix[r, :r])

        # Direct MST edge between r and j
        ivat_matrix[r, j] = vat_matrix[r, j]
        ivat_matrix[j, r] = ivat_matrix[r, j]

        # Update path-based distances to all other previous vertices c
        for c in range(r):
            if c != j:
                # Bottleneck distance: the worst edge on the path r->j->c
                ivat_matrix[r, c] = max(vat_matrix[r, j], ivat_matrix[j, c])
                ivat_matrix[c, r] = ivat_matrix[r, c]

    return ivat_matrix


def visualize_distance_matrix_large(
    distance_matrix, title="Distance Matrix Visualization", figure_size=(8, 6)
):
    """
    Visualizes a distance matrix as a heatmap.
    """
    plt.figure(figsize=figure_size)
    plt.imshow(distance_matrix, cmap="viridis", origin="upper")
    plt.colorbar(label="Distance")
    plt.title(title)
    plt.xlabel("Indices")
    plt.ylabel("Indices")
    plt.tight_layout()
    plt.show()

In [ ]:
# Post-Run Diversity Analysis
# ==================================================================

F_threshold = 45.0  # Fitness threshold for high-quality antibodies (same as GA)

analysis_report = analyze_antibody_diversity(
    final_population=ais_optimizer.population,
    top_n=100,
    F_threshold=F_threshold,
    R_peak=1,  # This radius might need tuning based on your distance values
)

# --- Print the human-readable summary ---
print("\n" + "=" * 40)
print("      AIS Post-Run Diversity Analysis")
print("=" * 40)
print(f"Analysis of Top {analysis_report['total_antibodies_analyzed']} Antibodies:")
print(f"Found {analysis_report['high_quality_count']} antibodies with score >= {F_threshold}.")
print(f"These antibodies form approximately {analysis_report['family_count']} distinct families.")
print("-" * 40)

if not analysis_report["families"]:
    print("No distinct families found above the threshold.")
else:
    for family in analysis_report["families"]:
        print(f"Antibody Family {family['family_id']}:")
        print(f"  - Top Score: {family['top_score']:.4f}")
        print(f"  - Members in Top 100: {family['member_count']}")

# --- Generate distance matrix and VAT/iVAT analysis ---
if analysis_report["high_quality_antibodies"]:
    high_quality_distance_matrix = generate_distance_matrix(
        analysis_report["high_quality_antibodies"]
    )

    # Compute the iVAT matrix
    vat_matrix, reorder = vat_reorder(high_quality_distance_matrix)
    ivat_matrix = iVAT_transform(vat_matrix)

    # Visualize the matrices
    visualize_distance_matrix_large(
        high_quality_distance_matrix, title="AIS Original Distance Matrix", figure_size=(4, 3)
    )
    visualize_distance_matrix_large(
        vat_matrix, title="AIS VAT Matrix Visualization", figure_size=(4, 3)
    )
    visualize_distance_matrix_large(
        ivat_matrix, title="AIS iVAT Matrix Visualization", figure_size=(4, 3)
    )

In [ ]:
# Plot High-Quality Antibodies
# ==================================================================


def plot_high_quality_antibodies(
    high_quality_antibodies, wavelengths, vertical_offset=0.1, figure_size=(10, 10)
):
    """
    Plots a given list of high-quality antibodies, typically generated
    from the analysis function.

    Args:
        high_quality_antibodies (list): A list of Antibody objects, sorted by fitness.
        wavelengths (np.array): The array of wavelength values for the x-axis.
        vertical_offset (float): The vertical spacing between plots.
        figure_size (tuple): The size of the matplotlib figure.
    """
    # The input is already a sorted list of high-quality Antibody objects
    num_antibodies = len(high_quality_antibodies)
    if num_antibodies == 0:
        print("No high-quality antibodies to plot.")
        return

    colors = plt.cm.viridis(np.linspace(0, 1, num_antibodies))  # Changed colormap for variety
    plt.figure(figsize=figure_size)

    # The list is already sorted descending, so we iterate normally
    # but plot them from bottom to top using their rank.
    for i, antibody_obj in enumerate(high_quality_antibodies):
        antibody = antibody_obj.genes
        basis_funcs = generate_basis_from_chromosome(antibody, wavelengths)

        # Plotting from bottom (rank N) to top (rank 1)
        plot_rank_offset = (num_antibodies - 1 - i) * vertical_offset

        for j, basis_func in enumerate(basis_funcs.T):
            scaled_basis_func = basis_func * 0.4  # Scale for better visualization
            plt.plot(
                wavelengths,
                scaled_basis_func + plot_rank_offset,
                color=colors[i],
                alpha=0.8,
                linewidth=2,
            )

    plt.title(f"Top {num_antibodies} High-Quality AIS Sensor Designs (Fitness >= Threshold)")
    plt.xlabel("Wavelength (µm)")
    plt.ylabel("Absorptivity (Offset Applied for Visualization)")

    # --- Dynamic Y-axis Ticks ---
    tick_interval = max(1, num_antibodies // 10)  # Aim for about 10 ticks
    tick_indices = [i for i in range(num_antibodies) if i % tick_interval == 0]
    if num_antibodies - 1 not in tick_indices:  # Ensure the last rank is always shown
        tick_indices.append(num_antibodies - 1)

    rank_labels = [f"Rank {i + 1}" for i in tick_indices]
    rank_offsets = [(num_antibodies - 1 - i) * vertical_offset for i in tick_indices]

    plt.yticks(rank_offsets, rank_labels)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()


if analysis_report["high_quality_antibodies"]:
    plot_high_quality_antibodies(
        analysis_report["high_quality_antibodies"], wavelengths, vertical_offset=0.5
    )

In [ ]:
# Plot Top 100 Solutions from Final Generation
# ==================================================================


def plot_top_100_last_gen(final_population, wavelengths, vertical_offset=0.1, figure_size=(10, 40)):
    """Plots the top 100 solutions from the final population."""

    # Sort by fitness (descending) and take the top 100
    top_solutions = sorted(final_population, key=lambda x: x.fitness, reverse=True)[:100]

    num_solutions = len(top_solutions)
    if num_solutions == 0:
        print("No solutions in the final generation to plot.")
        return

    colors = plt.cm.cividis(np.linspace(0, 1, num_solutions))
    plt.figure(figsize=figure_size)

    for i in reversed(range(num_solutions)):
        antibody = top_solutions[i]
        basis_funcs = generate_basis_from_chromosome(antibody.genes, wavelengths)
        rank = i
        for j, basis_func in enumerate(basis_funcs.T):
            scaled_basis_func = basis_func * 0.4
            plt.plot(
                wavelengths,
                scaled_basis_func + (num_solutions - 1 - rank) * vertical_offset,
                color=colors[rank],
                alpha=0.8,
                linewidth=2,
            )
    plt.title("Top 100 AIS Sensor Designs from Final Generation")
    plt.xlabel("Wavelength (µm)")
    plt.ylabel("Absorptivity (Offset Applied for Visualization)")

    tick_interval = 10
    rank_labels = [
        f"Rank {i + 1}"
        for i in range(num_solutions)
        if (i % tick_interval == 0) or i == num_solutions - 1
    ]
    rank_offsets = [
        (num_solutions - 1 - i) * vertical_offset
        for i in range(num_solutions)
        if (i % tick_interval == 0) or i == num_solutions - 1
    ]

    if 0 not in [i for i in range(num_solutions) if (i % tick_interval == 0)]:
        rank_labels.append("Rank 1")
        rank_offsets.append((num_solutions - 1) * vertical_offset)

    plt.yticks(rank_offsets, rank_labels)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()


# --- Call the plotting function ---
plot_top_100_last_gen(ais_optimizer.population, wavelengths, vertical_offset=0.5)

<h2 id="ais-export-and-compare">5. Export AIS Results and Compare with GA (optional)</h2>


In [ ]:
# Export AIS results
# ==================================================================
import json
from datetime import datetime

results_dir = "../output"
os.makedirs(results_dir, exist_ok=True)

ais_results = {
    "timestamp": datetime.now().isoformat(),
    "algorithm": "AIS",
    "best_fitness": float(best_fitness),
    "best_genes": [float(x) for x in best_genes.tolist()],
    "fitness_stats": {
        "best_per_gen": [float(x) for x in ais_optimizer.best_fitness_per_gen],
        "mean_per_gen": [float(x) for x in ais_optimizer.mean_fitness_per_gen],
        "worst_per_gen": [float(x) for x in ais_optimizer.worst_fitness_per_gen],
    },
}

ais_json_path = os.path.join(results_dir, "ais_results.json")
with open(ais_json_path, "w") as f:
    json.dump(ais_results, f, indent=2)

print(f"Saved AIS results to: {ais_json_path}")

# Optional: Export top-N solutions
TOP_N_EXPORT = 50
top_ais = sorted(ais_optimizer.population, key=lambda x: x.fitness, reverse=True)[:TOP_N_EXPORT]
export_rows = []
for rank, ab in enumerate(top_ais, start=1):
    row = {"rank": rank, "fitness": float(ab.fitness)}
    for i in range(4):
        row[f"mu_{i + 1}"] = float(ab.genes[i * 2])
        row[f"sigma_{i + 1}"] = float(ab.genes[i * 2 + 1])
    export_rows.append(row)

export_df = pd.DataFrame(export_rows)
export_csv_path = os.path.join(results_dir, "ais_top_solutions.csv")
export_df.to_csv(export_csv_path, index=False)
print(f"Saved AIS top-{TOP_N_EXPORT} solutions to: {export_csv_path}")

In [ ]:
# Optional GA vs AIS comparison (if GA JSON exists)
# ==================================================================

ga_json_path = os.path.join(results_dir, "ga_results.json")

if os.path.exists(ga_json_path):
    with open(ga_json_path, "r") as f:
        ga_results = json.load(f)
    print("\nGA results found. Comparing with AIS...")

    print(f"GA best fitness:  {ga_results.get('best_fitness'):.4f}")
    print(f"AIS best fitness: {ais_results.get('best_fitness'):.4f}")

    # Overlay best fitness per generation if available
    if "fitness_stats" in ga_results and "best_per_gen" in ga_results["fitness_stats"]:
        plt.figure(figsize=(10, 6))
        plt.plot(ga_results["fitness_stats"]["best_per_gen"], label="GA Best", color="purple")
        plt.plot(ais_optimizer.best_fitness_per_gen, label="AIS Best", color="green")
        plt.title("Best Fitness per Generation: GA vs AIS")
        plt.xlabel("Generation")
        plt.ylabel("Fitness")
        plt.legend()
        plt.grid(True, linestyle="--", alpha=0.6)
        plt.tight_layout()
        plt.show()
else:
    print(
        "\nNo GA results file found. To enable comparison, save GA results to '../output/ga_results.json'."
    )